In [ ]:
# ─── CELL 1: Load Model Once ──────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID     = "Qwen/Qwen2.5-Coder-32B-Instruct"
ADAPTER_PATH = "./java-vuln-adapter-32b-full"

tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model      = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Leverage base model reasoning by scaling down the adapter to 80%
model.set_adapter_scale("default", 0.8)

model.config.use_cache = True
model.eval()

DEVICE = next(model.parameters()).device
print(f"✅ Model ready on {DEVICE} (Adapter scale set to 0.8) — {torch.cuda.memory_allocated(0)/1e9:.1f} GB VRAM used")

In [ ]:
# ─── CELL 2: Test Your Java Code ──────────────────────────────────────────────
# Paste any complete Java class below and re-run this cell.
import sys

YOUR_CODE = """
import java.sql.Connection;
import java.sql.PreparedStatement;
import java.sql.ResultSet;
import java.sql.SQLException;
import java.util.logging.Logger;

public class CustomerSearchService {

    private static final Logger logger = Logger.getLogger(CustomerSearchService.class.getName());

    private final Connection connection;

    public CustomerSearchService(Connection connection) {
        this.connection = connection;
    }

    public ResultSet searchCustomers(String nameFilter, String region) throws SQLException {

        logger.info("Searching customers with filter: " + nameFilter + ", region: " + region);

        // Subtle SQL Injection: dynamic query building in WHERE clause
        String query = "SELECT id, name, email, region FROM customers WHERE 1=1 ";

        if (nameFilter != null && !nameFilter.isEmpty()) {
            query += " AND name LIKE '%" + nameFilter + "%' ";
        }
        if (region != null && !region.isEmpty()) {
            query += " AND region = '" + region + "' ";
        }
        PreparedStatement stmt = connection.prepareStatement(query);
        return stmt.executeQuery();
    }
}
"""

# Clear generation config overrides to avoid warnings
model.generation_config.max_new_tokens = 1000
model.generation_config.max_length = None

SYSTEM_PROMPT = """You are an expert Java security auditor. Analyze the provided code.
If the code is secure, output:
\"This Java code is completely secure and contains no vulnerabilities. No changes are required.\"

If the code is vulnerable, output your analysis in this exact format:
### 🛡️ Vulnerability Analysis
*   **Status**: VULNERABLE
*   **Type**: [Vulnerability Type]
*   **Severity**: HIGH

### 📝 Explanation
[Provide a brief explanation of the vulnerability]

### 🛠️ Fixed Code
```java
[Fixed complete Java code]
```"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": YOUR_CODE}
]

text     = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs   = tokenizer(text, return_tensors="pt").to(DEVICE)

print("🛡️  Analyzing code... (wait ~30 seconds)")
sys.stdout.flush()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Slice the generated tokens (excluding prompt) and move to CPU before decoding
generated_tokens = outputs[0][inputs.input_ids.shape[1]:].cpu()
result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(f"Done — Generated {len(generated_tokens)} tokens.")
print("\n🛡️  Security Analysis")
print("─" * 50)
print(result)
print("─" * 50)
